# MedMod in-hospital mortality: real-data runs on Google Colab

This notebook produced the real-data results in `mimic4_medmod_ihm_results.md`. The outputs are the ones printed during the runs. They contain only aggregates, such as stay counts, losses and test scores.

The runs need credentialed access to MIMIC-IV v1.0 and MIMIC-CXR-JPG 2.0.0. We kept the data in our own Google Drive. The run also needs about 20 GB of free Drive space, mostly for the ResNet-34 pretraining checkpoints. Our Drive filled up during the run, so some ResNet-34 pretraining checkpoints were not uploaded. Before the probes ran, we copied them back to their folders from Colab's local Drive cache, where they had been kept. The code is a zip of this repository made by `examples/medmod_runners/colab_bundle.py`, and the setup cell prints when the zip was made.

Before the first run, make the upload files on the machine that has the data. `python -m examples.medmod_runners.colab_bundle --data-root <mimic_data> --ihm --medfuse-weights <folder>` writes `convirt_data.zip`, `ihm_data.zip` and `convirt_code.zip` to `<mimic_data>/colab_upload/`. `python -m pyhealth.datasets.medmod.medsiglip --images <resized images> --out medsiglip_448.npz` writes the MedSigLIP embeddings. Upload these four files to `My Drive/medmod_convirt/`. Without a `pyhealth_cache.tar` there, the setup cell prints a tar error and the first run builds PyHealth's cache.

The notebook runs two kinds of scripts.

- **The example scripts** `mimic4_medmod_ihm_early_fusion.py` and `mimic4_medmod_convirt.py` run once each, to check that they give the same scores as the runners.
- **The runner scripts** in `examples/medmod_runners/` train the same models with the same settings and produce the five-seed results. They also compute bootstrap confidence intervals, write a JSON report for every run and save checkpoints, so that a run continues after a disconnect.

Every model runs with MedMod's five seeds. As the paper describes, each ConVIRT pretraining run is probed on the epoch with the highest validation AUROC. For this choice, a quick probe (features computed once, no augmentation) runs on every 10th pretraining epoch. The normal probe then runs on the chosen epoch.

Run the cells from top to bottom. The cell "Run all jobs in parallel" runs every job as a background process, several at a time, and each job writes its full output to `logs/<job>.log` in the results folder. After a disconnect, run the cells again. Finished jobs are skipped, and unfinished ones continue from their last checkpoint.

In [35]:
# Setup (run every session)
from google.colab import drive
drive.mount('/content/drive')
import os, json
DRIVE = '/content/drive/MyDrive/medmod_convirt'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# copy to the local disk (many small files are slow on Drive)
!test -d /content/data/medmod_extracted || (unzip -q -n {DRIVE}/convirt_data.zip -d /content/data && echo pretraining data unpacked)
!test -d /content/data/mimiciv_medmod_slim || (unzip -q -n {DRIVE}/ihm_data.zip -d /content/data && echo IHM data unpacked)
!test -d /content/data/pyhealth_cache || tar -xf {DRIVE}/pyhealth_cache.tar -C /content/data
!rm -rf /content/code && unzip -q {DRIVE}/convirt_code.zip -d /content && echo code unpacked
!pip install -q -e /content/code
CODE = json.load(open('/content/code/CODE_VERSION.json'))
print('code zip made', CODE['created'])
RESULTS = f'{DRIVE}/medmod_ihm_results'; os.makedirs(RESULTS, exist_ok=True)
W = '/content/data/medfuse_weights'
EMB = f'{DRIVE}/medsiglip_448.npz'
DATA = '--ehr-root /content/data/mimiciv_medmod_slim/1.0 --cxr-root /content/data/mimic-cxr-jpg/2.0.0 --image-dir /content/data/mimic-cxr-jpg/2.0.0/resized --cache-dir /content/data/pyhealth_cache'
SEEDS = [1002, 2918, 5793, 7261, 84305]  # MedMod's seeds

def finished(name, key):
    """True if the runner report RESULTS/<name>.json has the section `key`."""
    f = f'{RESULTS}/{name}.json'
    return os.path.exists(f) and key in json.load(open(f))

def pretraining_done(name):
    """True if the pretraining report RESULTS/<name>.json has all its epochs."""
    f = f'{RESULTS}/{name}.json'
    if not os.path.exists(f):
        return False
    r = json.load(open(f))
    return r['epochs_done'] >= r['settings']['epochs']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NVIDIA A100-SXM4-40GB, 40960 MiB
code unpacked
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pyhealth (pyproject.toml) ... done
code zip made 2026-11-23T00:34:55+00:00


In [36]:
# Check the MedSigLIP embeddings file
import sys; sys.path.insert(0, '/content/code')
from pyhealth.datasets.medmod.medsiglip import load_embeddings
index, emb, meta = load_embeddings(EMB)
print(len(index), emb.shape, meta['revision'][:8])

12625 (12625, 1152) 9cea28a1


## Jobs

In [37]:
# Jobs: every run of this notebook, with what it needs and how to tell it is done
import glob
THREADS = 2  # CPU threads for each job's own math, so that no job takes over the shared CPU
PY = f'cd /content/code && OMP_NUM_THREADS={THREADS} MKL_NUM_THREADS={THREADS} python -u -W ignore'
RUN = f'--data-root /content/data --results-dir {RESULTS}'
EHR_W = f'--ehr-weights {W}/uni_ehr_all/best_checkpoint.pth.tar'
CXR_W = f'--cxr-weights {W}/cxr_rad_full/best_checkpoint.pth.tar'
LOGS = f'{RESULTS}/logs'; os.makedirs(LOGS, exist_ok=True)

SEEDS_HERE = SEEDS   # the seeds this session runs; e.g. [1002, 2918] here and the rest in a second session
RUN_CHECKS = True    # the two example scripts and the seed-0 runner check (run them in one session only)

def job_logs(name):
    """The job's log, and the copies Google Drive makes ("name (1).log") when
    the same file is written from another session."""
    return sorted(glob.glob(f'{LOGS}/{glob.escape(name)}*.log'))

def log_has(name, text):
    return any(text in open(f).read() for f in job_logs(name))

# candidate pretraining epochs for the paper's selection: every 10th epoch
CANDIDATES = list(range(9, 300, 10))

def chosen_epoch(enc, s):
    """The candidate epoch whose quick probe has the highest validation AUROC (validation only)."""
    best = {}
    for e in CANDIDATES:
        r = json.load(open(f'{RESULTS}/select_{enc}_seed{s}_epoch{e:03d}.json'))
        best[e] = max(row['val_auroc'] for row in r['epochs'])
    return max(CANDIDATES, key=lambda e: (best[e], -e))  # ties go to the earlier epoch

jobs = []
def job(name, command, images, done, needs=(), heavy=False):
    """command and done are functions, evaluated when the job starts and ends.
    heavy marks a ResNet-34 pretraining run, which needs a lot of GPU memory."""
    jobs.append({'name': name, 'command': command, 'images': images, 'done': done,
                 'needs': list(needs), 'heavy': heavy})

if RUN_CHECKS:
    job('early_fusion_example', lambda: f'{PY} examples/mimic4_medmod_ihm_early_fusion.py {DATA} {EHR_W} {CXR_W}',
        True, lambda: log_has('early_fusion_example', 'RESULT'))
    job('early_fusion_resnet34_seed0', lambda: f'{PY} -m examples.medmod_runners.early_fusion_run {RUN} {EHR_W} {CXR_W} --tag early_fusion_resnet34_seed0',
        True, lambda: finished('early_fusion_resnet34_seed0', 'test'))
for s in SEEDS_HERE:
    t = f'early_fusion_resnet34_seed{s}'
    job(t, lambda t=t, s=s: f'{PY} -m examples.medmod_runners.early_fusion_run {RUN} {EHR_W} {CXR_W} --seed {s} --tag {t}',
        True, lambda t=t: finished(t, 'test'))
    t = f'early_fusion_medsiglip_seed{s}'
    job(t, lambda t=t, s=s: f'{PY} -m examples.medmod_runners.early_fusion_run {RUN} {EHR_W} --embeddings {EMB} --seed {s} --tag {t}',
        False, lambda t=t: finished(t, 'test'))
    for enc, images, extra in [('resnet34', True, ''), ('medsiglip', False, f'--embeddings {EMB}')]:
        pre, probe = f'convirt_{enc}_seed{s}', f'probe_{enc}_seed{s}'
        # keeps the weights of every 10th epoch (the candidates)
        job(pre, lambda pre=pre, s=s, extra=extra: f'{PY} -m examples.medmod_runners.convirt_pretrain {RUN} --tag {pre} --seed {s} --keep-every 10 {extra}',
            images, lambda pre=pre: pretraining_done(pre), heavy=images)
        # the paper's selection: a quick probe (features computed once, no augmentation) on each
        # candidate epoch; the epoch with the highest validation AUROC is chosen
        selects = []
        for e in CANDIDATES:
            sel = f'select_{enc}_seed{s}_epoch{e:03d}'
            selects.append(sel)
            job(sel, lambda pre=pre, sel=sel, s=s, e=e, extra=extra: f'{PY} -m examples.medmod_runners.convirt_probe {RUN} --checkpoint {RESULTS}/{pre}/epoch_{e:03d}.pt {extra} --no-augment --seed {s} --tag {sel}',
                images, lambda sel=sel: finished(sel, 'test_best_val'), needs=[pre])
        # the normal probe (MedMod's settings, with augmentation) on the chosen epoch
        job(probe, lambda pre=pre, probe=probe, enc=enc, s=s, extra=extra: f'{PY} -m examples.medmod_runners.convirt_probe {RUN} --checkpoint {RESULTS}/{pre}/epoch_{chosen_epoch(enc, s):03d}.pt {extra} --seed {s} --tag {probe}',
            images, lambda probe=probe: finished(probe, 'test_final'), needs=selects)
if RUN_CHECKS and 84305 in SEEDS_HERE:
    for enc, images, extra in [('resnet34', True, ''), ('medsiglip', False, f'--embeddings {EMB}')]:
        name = f'convirt_example_probe_{enc}'
        # the example script on the same chosen epoch as the runner's seed-84305 probe, to compare
        job(name, lambda enc=enc, extra=extra: f'{PY} examples/mimic4_medmod_convirt.py probe {DATA} {extra} --checkpoint {RESULTS}/convirt_{enc}_seed84305/epoch_{chosen_epoch(enc, 84305):03d}.pt',
            images, lambda name=name: log_has(name, 'TEST'),
            needs=[f'select_{enc}_seed84305_epoch{e:03d}' for e in CANDIDATES])
print(len(jobs), 'jobs,', sum(not j['done']() for j in jobs), 'still to run')

334 jobs, 164 still to run


In [46]:
# Run all jobs in parallel (waits until every job is done; stop the cell to stop all jobs)
import signal, subprocess, time

MAX_JOBS = 6        # jobs at the same time
MAX_IMAGE_JOBS = 4  # of those, jobs that decode X-ray images (the ResNet-34 runs). A100: 4. T4: 1.
MAX_HEAVY_JOBS = 2  # of those, ResNet-34 pretraining runs (about 14 GB of GPU memory each). A100: 2. T4: 1.

RETRIES = 2         # a failed job is started again up to this many times (e.g. after a Drive hiccup)

def result_line(name):
    """The last line of the job's logs that reports a result."""
    keys = ('RESULT', 'BEST', 'TEST final', 'TEST {', 'done ')
    lines = [l.strip() for f in job_logs(name) for l in open(f) if l.startswith(keys)]
    return lines[-1] if lines else '(no result line)'

def run_jobs(jobs):
    done = {j['name'] for j in jobs if j['done']()}
    waiting = [j for j in jobs if j['name'] not in done]
    running, failed, last_status, tries = {}, set(), 0, {}
    try:
        while waiting or running:
            for name, (proc, j) in list(running.items()):
                if proc.poll() is None:
                    continue
                del running[name]
                if j['done']():
                    done.add(name)
                    if name.startswith('select_'):  # many short jobs: one compact line each
                        r = json.load(open(f'{RESULTS}/{name}.json'))
                        print(time.strftime('%H:%M'), 'finished', name, '| best validation AUROC',
                              round(max(row['val_auroc'] for row in r['epochs']), 4), flush=True)
                    else:
                        print(time.strftime('%H:%M'), 'finished', name, '|', result_line(name)[:300], flush=True)
                elif tries.get(name, 0) < RETRIES:
                    tries[name] = tries.get(name, 0) + 1
                    waiting.insert(0, j)
                    print(time.strftime('%H:%M'), 'retrying', name, f'(exit {proc.returncode}, try {tries[name] + 1})', flush=True)
                else:
                    failed.add(name)
                    print(time.strftime('%H:%M'), 'FAILED', name, f'(exit {proc.returncode}); see logs/{name}.log', flush=True)
            for j in list(waiting):
                if any(n in failed for n in j['needs']):
                    waiting.remove(j); failed.add(j['name'])
                    print('skipped', j['name'], '(a job it needs failed)', flush=True)
                    continue
                if len(running) >= MAX_JOBS or not all(n in done for n in j['needs']):
                    continue
                if j['images'] and sum(r[1]['images'] for r in running.values()) >= MAX_IMAGE_JOBS:
                    continue
                if j['heavy'] and sum(r[1]['heavy'] for r in running.values()) >= MAX_HEAVY_JOBS:
                    continue
                log = open(f"{LOGS}/{j['name']}.log", 'a')
                proc = subprocess.Popen(j['command'](), shell=True, stdout=log,
                                        stderr=subprocess.STDOUT, start_new_session=True)
                running[j['name']] = (proc, j); waiting.remove(j)
                print(time.strftime('%H:%M'), 'started', j['name'], flush=True)
            if time.time() - last_status > 1800:  # a status line every 30 minutes
                last_status = time.time()
                print(time.strftime('%H:%M'), f'running {sorted(running)}, {len(done)} done, {len(waiting)} waiting', flush=True)
            time.sleep(20)
    finally:  # stopping the cell stops the jobs too
        for proc, _ in running.values():
            os.killpg(proc.pid, signal.SIGTERM)
    print('all done' if not failed else f'failed: {sorted(failed)}. Run this cell again to retry them.')

run_jobs(jobs)

22:24 started select_resnet34_seed5793_epoch129
22:24 started select_resnet34_seed5793_epoch139
22:24 started select_resnet34_seed5793_epoch149
22:24 started select_resnet34_seed5793_epoch159
22:24 running ['select_resnet34_seed5793_epoch129', 'select_resnet34_seed5793_epoch139', 'select_resnet34_seed5793_epoch149', 'select_resnet34_seed5793_epoch159'], 272 done, 58 waiting
22:25 finished select_resnet34_seed5793_epoch129 | best validation AUROC 0.8362
22:25 finished select_resnet34_seed5793_epoch139 | best validation AUROC 0.8261
22:25 finished select_resnet34_seed5793_epoch149 | best validation AUROC 0.8344
22:25 finished select_resnet34_seed5793_epoch159 | best validation AUROC 0.823
22:25 started select_resnet34_seed5793_epoch169
22:25 started select_resnet34_seed5793_epoch179
22:25 started select_resnet34_seed5793_epoch189
22:25 started select_resnet34_seed5793_epoch199
22:26 finished select_resnet34_seed5793_epoch169 | best validation AUROC 0.8306
22:26 finished select_resnet34_s

In [47]:
# Result line of every job (from its log)
for j in jobs:
    if j['name'].startswith('select_'):
        continue
    if job_logs(j['name']):
        print(f"{j['name']}: {result_line(j['name'])[:300]}")

early_fusion_example: RESULT {"test_auroc": 0.8408456706815899, "test_auprc": 0.5136693875944193}
early_fusion_resnet34_seed0: BEST {"epoch": 49, "val_auroc": 0.8462187996082008} TEST {"n": 1241, "deaths": 187, "auroc": 0.8408456706815899, "auroc_ci95": [0.8108897467691377, 0.8675122067693608], "auprc": 0.5136693875944193, "auprc_ci95": [0.44424327305633265, 0.5885279757236426]}
early_fusion_resnet34_seed1002: BEST {"epoch": 23, "val_auroc": 0.8469280913297531} TEST {"n": 1241, "deaths": 187, "auroc": 0.8405057382621843, "auroc_ci95": [0.8104729171239627, 0.8678245842452091], "auprc": 0.5138196265328214, "auprc_ci95": [0.44384553768414287, 0.5887811958100828]}
early_fusion_medsiglip_seed1002: BEST {"epoch": 15, "val_auroc": 0.857128381801601} TEST {"n": 1241, "deaths": 187, "auroc": 0.8447219149864533, "auroc_ci95": [0.817966798482145, 0.8701328849748894], "auprc": 0.5329742379685934, "auprc_ci95": [0.467682496763085, 0.6067810449977399]}
convirt_resnet34_seed1002: done {"epochs_done":

## Summary

In [48]:
# Pretraining summary, both encoders and all seeds (reads the saved epoch logs)
for name, prefix in [('ResNet-34', 'convirt_resnet34'), ('MedSigLIP', 'convirt_medsiglip')]:
    for s in SEEDS:
        f = f'{RESULTS}/{prefix}_seed{s}_epochs.jsonl'
        if not os.path.exists(f):
            print(f'{name}, seed {s}: not run yet')
            continue
        rows = [json.loads(line) for line in open(f)]
        best = min(rows, key=lambda r: r['val_loss'])
        last = rows[-1]
        print(f"{name}, seed {s}: {len(rows)} epochs. Lowest validation loss {best['val_loss']:.2f} "
              f"at epoch {best['epoch']}. Last epoch {last['epoch']}: validation loss {last['val_loss']:.2f}.")

ResNet-34, seed 1002: 300 epochs. Lowest validation loss 8.89 at epoch 135. Last epoch 299: validation loss 13.48.
ResNet-34, seed 2918: 300 epochs. Lowest validation loss 9.03 at epoch 110. Last epoch 299: validation loss 15.20.
ResNet-34, seed 5793: 300 epochs. Lowest validation loss 8.98 at epoch 125. Last epoch 299: validation loss 13.42.
ResNet-34, seed 7261: 300 epochs. Lowest validation loss 8.95 at epoch 127. Last epoch 299: validation loss 14.59.
ResNet-34, seed 84305: 300 epochs. Lowest validation loss 8.84 at epoch 138. Last epoch 299: validation loss 12.80.
MedSigLIP, seed 1002: 300 epochs. Lowest validation loss 8.68 at epoch 17. Last epoch 299: validation loss 47.20.
MedSigLIP, seed 2918: 300 epochs. Lowest validation loss 8.64 at epoch 27. Last epoch 299: validation loss 43.55.
MedSigLIP, seed 5793: 300 epochs. Lowest validation loss 8.79 at epoch 21. Last epoch 299: validation loss 46.32.
MedSigLIP, seed 7261: 300 epochs. Lowest validation loss 8.73 at epoch 21. Last ep

In [49]:
# Summary of the runner reports (reads the JSON reports in RESULTS, aggregates only)
import numpy as np

def reports(names):
    return [json.load(open(f'{RESULTS}/{n}.json')) for n in names
            if os.path.exists(f'{RESULTS}/{n}.json')]

def seed_summary(name, runs, section):
    runs = [r for r in runs if section in r]  # finished runs only
    print(f'{name}: {len(runs)} seeds')
    for r in runs:
        t = r[section]
        print(f"  seed {r['settings']['seed']:>6}  "
              f"AUROC {t['auroc']:.4f} ({t['auroc_ci95'][0]:.4f} to {t['auroc_ci95'][1]:.4f})  "
              f"AUPRC {t['auprc']:.4f} ({t['auprc_ci95'][0]:.4f} to {t['auprc_ci95'][1]:.4f})")
    for m in ('auroc', 'auprc'):
        v = np.array([r[section][m] for r in runs])
        sd = v.std(ddof=1) if len(v) > 1 else float('nan')  # sample standard deviation
        print(f'  {m.upper()} mean {v.mean():.4f} +- {sd:.4f} (sample std)')
    return {r['settings']['seed']: r[section] for r in runs}

def compare(a, b):
    both = sorted(set(a) & set(b))  # paired by seed
    for m in ('auroc', 'auprc'):
        d = np.array([b[s][m] - a[s][m] for s in both])
        if len(d) > 1:
            print(f'  MedSigLIP - ResNet-34 {m.upper()}: {d.mean():+.4f} +- {d.std(ddof=1):.4f}, higher in {(d > 0).sum()}/{len(d)} seeds')

print('Paper: Early Fusion (Table 3) AUROC 0.842 +- 0.004, AUPRC 0.515 +- 0.020; ConVIRT linear probing (Table 4) AUROC 0.847 +- 0.002, AUPRC 0.482 +- 0.016; Table 8 0.813 +- 0.010\n')
ef_r = seed_summary('Early Fusion, ResNet-34', reports([f'early_fusion_resnet34_seed{s}' for s in SEEDS]), 'test')
ef_m = seed_summary('Early Fusion, MedSigLIP', reports([f'early_fusion_medsiglip_seed{s}' for s in SEEDS]), 'test')
compare(ef_r, ef_m)
print()
for enc in ('resnet34', 'medsiglip'):
    picks = {}
    for s in SEEDS:
        try:
            picks[s] = chosen_epoch(enc, s)
        except FileNotFoundError:
            picks[s] = 'not ready'
    print(f'Chosen pretraining epoch by validation AUROC, {enc}:', picks)
cv_r = seed_summary('ConVIRT probe on the chosen epoch, ResNet-34', reports([f'probe_resnet34_seed{s}' for s in SEEDS]), 'test_final')
cv_m = seed_summary('ConVIRT probe on the chosen epoch, MedSigLIP', reports([f'probe_medsiglip_seed{s}' for s in SEEDS]), 'test_final')
compare(cv_r, cv_m)
print()
for r in reports(['early_fusion_resnet34_seed0']):
    t = r['test']
    print(f"Early Fusion runner, ResNet-34, seed 0: AUROC {t['auroc']:.4f}, AUPRC {t['auprc']:.4f}")

Paper: Early Fusion (Table 3) AUROC 0.842 +- 0.004, AUPRC 0.515 +- 0.020; ConVIRT linear probing (Table 4) AUROC 0.847 +- 0.002, AUPRC 0.482 +- 0.016; Table 8 0.813 +- 0.010

Early Fusion, ResNet-34: 5 seeds
  seed   1002  AUROC 0.8405 (0.8105 to 0.8678)  AUPRC 0.5138 (0.4438 to 0.5888)
  seed   2918  AUROC 0.8410 (0.8110 to 0.8682)  AUPRC 0.5148 (0.4451 to 0.5896)
  seed   5793  AUROC 0.8410 (0.8110 to 0.8682)  AUPRC 0.5140 (0.4437 to 0.5892)
  seed   7261  AUROC 0.8407 (0.8105 to 0.8683)  AUPRC 0.5144 (0.4450 to 0.5895)
  seed  84305  AUROC 0.8401 (0.8104 to 0.8675)  AUPRC 0.5128 (0.4446 to 0.5881)
  AUROC mean 0.8407 +- 0.0004 (sample std)
  AUPRC mean 0.5140 +- 0.0007 (sample std)
Early Fusion, MedSigLIP: 5 seeds
  seed   1002  AUROC 0.8447 (0.8180 to 0.8701)  AUPRC 0.5330 (0.4677 to 0.6068)
  seed   2918  AUROC 0.8464 (0.8195 to 0.8721)  AUPRC 0.5368 (0.4702 to 0.6109)
  seed   5793  AUROC 0.8461 (0.8185 to 0.8717)  AUPRC 0.5353 (0.4700 to 0.6084)
  seed   7261  AUROC 0.8474 (0.81

In [50]:
# Files in the results folder
!ls {RESULTS}

 convirt_medsiglip_seed1002
 convirt_medsiglip_seed1002_epochs.jsonl
 convirt_medsiglip_seed1002.json
 convirt_medsiglip_seed2918
 convirt_medsiglip_seed2918_epochs.jsonl
 convirt_medsiglip_seed2918.json
 convirt_medsiglip_seed5793
 convirt_medsiglip_seed5793_epochs.jsonl
 convirt_medsiglip_seed5793.json
 convirt_medsiglip_seed7261
 convirt_medsiglip_seed7261_epochs.jsonl
 convirt_medsiglip_seed7261.json
 convirt_medsiglip_seed84305
 convirt_medsiglip_seed84305_epochs.jsonl
 convirt_medsiglip_seed84305.json
 convirt_resnet34_seed1002
 convirt_resnet34_seed1002_epochs.jsonl
 convirt_resnet34_seed1002.json
 convirt_resnet34_seed2918
 convirt_resnet34_seed2918_epochs.jsonl
 convirt_resnet34_seed2918.json
 convirt_resnet34_seed5793
 convirt_resnet34_seed5793_epochs.jsonl
 convirt_resnet34_seed5793.json
 convirt_resnet34_seed7261
 convirt_resnet34_seed7261_epochs.jsonl
 convirt_resnet34_seed7261.json
 convirt_resnet34_seed84305
 convirt_resnet34_seed84305_epochs.jsonl
 convirt_resnet34_seed